# FlowCast — EDA of the raw data (decision notebook)

**Purpose:** look at the raw USGS river readings and Open-Meteo weather *before* building any
training data, and turn what we find into a short list of decisions. The new training-data
script should do exactly what the **Decision** lines say — nothing added "just because."

**How every section works**
1. **Question** — one plain-language question.
2. **Table** — a small table that answers it.
3. **How to read it** — what to look at in the table.
4. **Decision** — what we will do in the training data because of it.

**The one rule: decisions use 2020–2024 only.**
2025-onward is our test period. If we choose cut-offs or cleaning rules while looking at test
data, the test score stops being a fair "unseen data" check. So every table that drives a
decision is filtered to `date < 2025-01-01` (`train` below). The full data is only used to
describe what we have.

**Needs:** pandas only (no plotting libraries). Run it with the **Python (flowcast_env)** kernel.

In [1]:
import os, sys
from pathlib import Path

# Run from the repo root no matter where Jupyter was started.
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

SOURCE = "s3"   # "s3" = newest copies in the S3 bucket; "local" = the data/ folder on this computer
os.environ["FLOWCAST_DATA_SOURCE"] = SOURCE

import numpy as np
import pandas as pd
from src.flowcast.storage import read_dataset

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

SPLIT_DATE = pd.Timestamp("2025-01-01")    # train = before this date, test = on/after
TRAIN_START = pd.Timestamp("2020-01-01")
TRAIN_DAYS = (SPLIT_DATE - TRAIN_START).days   # 1,827 calendar days in 2020-2024

# Thresholds used below. Changing one here changes it everywhere.
MIN_COMPLETE_PCT = 90     # keep a gauge's measure only if it has readings on >= 90% of training days
BIG_JUMP = 20             # "suspicious" = today's flow is 20x or more yesterday's
RAINY_DAY_IN = 0.25       # inches of rain (today + yesterday) that counts as a rainy day
BIG_RAIN_IN = 1.0         # inches in one day that counts as a big storm
STUCK_DAYS = 7            # same exact reading this many days in a row = "stuck?"
LOW_PCT, HIGH_PCT = 10, 90   # drought = bottom 10% of the gauge's history, high water = top 10%

In [2]:
ID = {"monitoring_location_id": str}
usgs    = read_dataset("usgs_daily", dtype=ID)
weather = read_dataset("weather_daily")
cells   = read_dataset("weather_cell_map", dtype=ID)
sites   = read_dataset("usgs_active_sites", dtype=ID)

usgs["date"] = pd.to_datetime(usgs["time"])
weather["date"] = pd.to_datetime(weather["date"])
usgs["measure"] = usgs["variable"].map({"streamflow_cfs_mean": "streamflow", "gauge_height_ft_mean": "gauge_height"})

train = usgs[usgs["date"] < SPLIT_DATE].copy()          # <- decisions use this
train_weather = weather[weather["date"] < SPLIT_DATE].copy()

print(f"USGS rows:    {len(usgs):,}  ({usgs['date'].min().date()} to {usgs['date'].max().date()})")
print(f"Weather rows: {len(weather):,}  ({weather['date'].min().date()} to {weather['date'].max().date()})")
print(f"Training USGS rows (2020-2024): {len(train):,}")

  reading s3://flowcast-pred-app-data/raw/usgs/daily_values/2026-10-25/usgs_daily_2020_present.csv
  reading s3://flowcast-pred-app-data/raw/openmeteo/daily_weather/2026-10-25/weather_daily_2020_present.csv
  reading s3://flowcast-pred-app-data/raw/openmeteo/site_cell_map/2026-10-25/site_weather_cell_map.csv
  reading s3://flowcast-pred-app-data/raw/usgs/active_sites/2026-10-25/usgs_active_sites.csv
USGS rows:    432,444  (2020-01-01 to 2026-10-25)
Weather rows: 141,873  (2020-01-01 to 2026-10-24)
Training USGS rows (2020-2024): 316,166


## 1. What do we have?

**Question:** which gauges are there, what does each one measure, and over what dates?

Each USGS gauge can record two things (both as a **daily mean** — USGS publishes no daily max here):
- **streamflow** (cubic feet per second, cfs) — how much water passes by. Comparable *within* a river over time.
- **gauge height** (feet) — how high the water surface is above a local reference mark. The reference
  mark is different at every gauge, so 3 ft at one gauge means nothing at another.

In [3]:
overview = (usgs.groupby("measure")
            .agg(gauges=("monitoring_location_id", "nunique"), rows=("value", "size"),
                 first_date=("date", "min"), last_date=("date", "max"))
            .assign(first_date=lambda d: d["first_date"].dt.date, last_date=lambda d: d["last_date"].dt.date))
display(overview)

both = sites.groupby(["has_streamflow", "has_gauge_height"]).size().rename("gauges").reset_index()
print("Gauges by what they measure:")
display(both)
print(f"Total gauges: {sites['monitoring_location_id'].nunique()}  |  "
      f"counties: {sites['county_name'].nunique()}  |  weather grid squares: {weather['cell_id'].nunique()}")

,gauges,rows,first_date,last_date
measure,,,,
gauge_height,101,235699,2020-01-01,2026-10-24
streamflow,85,196745,2020-01-01,2026-10-25


Gauges by what they measure:


,has_streamflow,has_gauge_height,gauges
0,False,True,17
1,True,False,1
2,True,True,84


Total gauges: 102  |  counties: 16  |  weather grid squares: 57


In [4]:
# Data-quality labels USGS attaches to readings (whole period, just to describe).
print("Approval status:")
display(usgs.groupby([usgs["date"] < SPLIT_DATE, "approval_status"]).size()
        .rename("rows").rename_axis(["in_training_period", "approval_status"]).reset_index())
print("Qualifiers (most rows have none):")
display(usgs["qualifier"].fillna("(none)").value_counts().rename("rows").to_frame())

Approval status:


,in_training_period,approval_status,rows
0,False,Approved,92193
1,False,Provisional,24085
2,True,Approved,316026
3,True,Provisional,140


Qualifiers (most rows have none):


,rows
qualifier,
(none),417805
['REVISED'],11239
['ESTIMATED'],3082
"['ESTIMATED', 'REVISED']",118
['FORCEINTERPOLATION'],78
['REGULATED'],75
"['ESTIMATED', 'REGULATED']",28
"['ESTIMATED', 'ICE']",6
['UNKNOWNREGULATION'],5


**How to read it:** "Provisional" readings can still be revised by USGS; they should only appear
in recent months. Qualifiers like `ESTIMATED` (USGS filled the value in, e.g. a broken sensor) and
`REGULATED` (a dam controls the flow) are a small share of rows.

**Decision 1:** keep both measures for now (section 2 picks one per gauge). Keep provisional and
qualified readings — they are a tiny share and `daily_update.py` already re-pulls the last 30
days so USGS corrections replace provisional values.

## 2. Is it complete? (training years only)

**Question:** for each gauge and measure, on what share of the 1,827 training days (2020–2024) is
there a reading?

**Worked example:** a gauge with readings on 1,700 of 1,827 days is 1,700 ÷ 1,827 = **93%** complete.
A gauge installed in mid-2023 can be at most ~35% complete, because the days before it existed count as missing.

In [5]:
comp = (train.groupby(["monitoring_location_id", "measure"])["value"].count()
        .div(TRAIN_DAYS).mul(100).rename("pct_days").reset_index()
        .pivot(index="monitoring_location_id", columns="measure", values="pct_days"))
comp = comp.reindex(sites["monitoring_location_id"])   # include gauges with no training data at all

bands = [-0.1, 50, 75, 90, 99, 100]
labels = ["0-50%", "50-75%", "75-90%", "90-99%", "99-100%"]
print("How many gauges fall in each completeness band (NaN = measure not recorded at that gauge):")
display(pd.DataFrame({m: pd.cut(comp[m], bands, labels=labels).value_counts(dropna=False).sort_index()
                      for m in ["streamflow", "gauge_height"]}))

How many gauges fall in each completeness band (NaN = measure not recorded at that gauge):


,streamflow,gauge_height
0-50%,5,8
50-75%,0,0
75-90%,1,0
90-99%,6,21
99-100%,71,72
NaN,19,1


In [6]:
# Pick ONE measure per gauge: streamflow if it is complete enough, otherwise gauge height, otherwise drop.
ok_flow = comp["streamflow"] >= MIN_COMPLETE_PCT
ok_gh   = comp["gauge_height"] >= MIN_COMPLETE_PCT
comp["use_measure"] = np.select([ok_flow, ok_gh], ["streamflow", "gauge_height"], default="drop")

display(comp["use_measure"].value_counts().rename("gauges").to_frame())
print("Gauges that would be dropped (and why):")
display(comp[comp["use_measure"] == "drop"]
        .join(sites.set_index("monitoring_location_id")[["monitoring_location_name", "first_date"]])
        .round(1))

,gauges
use_measure,
streamflow,77
gauge_height,17
drop,8


Gauges that would be dropped (and why):


,gauge_height,streamflow,use_measure,monitoring_location_name,first_date
monitoring_location_id,,,,,
USGS-0214645075,14.90,15.00,drop,"TRIB TO BRIAR CREEK AT COLONY RD AT CHARLOTTE, NC",2020-01-01
USGS-0214645080,15.00,15.00,drop,"TRIB TO BRIAR CR AT RUNNYMEDE LN AT CHARLOTTE, NC",2020-01-01
USGS-021473470,34.40,35.00,drop,"SOUTH FORK FISHING CREEK BELOW McCONNELLS,SC",2020-01-01
USGS-021474095,35.30,NaN,drop,"GREAT FALLS RESERVOIR TAILRACE BL GREAT FALLS, SC",2023-03-28
USGS-021474750,7.50,NaN,drop,"ROCKY CREEK AT SC 901 ABOVE GREAT FALLS, SC",2024-08-15
USGS-0215329875,10.30,10.30,drop,"BUFFALO CREEK BL MOSS LAKE DAM NR OAK GROVE, NC",2024-06-26
USGS-02123629,47.30,47.40,drop,"LITTLE MOUNTAIN CREEK AT NC HWY 740 AT BADIN, NC",2022-08-19
USGS-02130450,34.30,NaN,drop,"THOMPSON CREEK AT CHESTERFIELD, SC",2020-10-09


**How to read it:** most gauges are 99–100% complete. The "dropped" table shows the few that are
not — usually because they were installed partway through 2020–2024 (check `first_date`).

**Why prefer streamflow:** it is the measure USGS's own "below/above normal" categories use, and it
has a meaning that does not depend on where the reference mark was set.

**Decision 2:** one measure per gauge — streamflow if ≥ 90% complete in 2020–2024, else gauge height
if ≥ 90%, else drop the gauge. Saved as `use_measure` and written to the decisions file at the end.

In [7]:
# From here on: one series per kept gauge, the "level", using the chosen measure.
keep = comp[comp["use_measure"] != "drop"]["use_measure"]
level = (usgs.merge(keep.rename("use_measure").reset_index(), on="monitoring_location_id")
         .query("measure == use_measure")[["monitoring_location_id", "measure", "date", "value"]]
         .sort_values(["monitoring_location_id", "date"]).reset_index(drop=True))
level_train = level[level["date"] < SPLIT_DATE].copy()
print(f"Kept gauges: {level['monitoring_location_id'].nunique()}  |  training rows: {len(level_train):,}")

Kept gauges: 94  |  training rows: 171,099


## 3. Is it believable? (training years only)

**Question:** are there readings that look like sensor errors rather than real river behavior?

Three checks:
- **(a) Impossible values** — negative streamflow or zero flow. (Negative *gauge height* is fine:
  it just means the water is below the gauge's reference mark.)
- **(b) Stuck readings** — the exact same value 7+ days in a row. Could be a frozen sensor, or a
  real dry spell where a tiny creek sits at 0.02 cfs.
- **(c) Huge one-day jumps** — today ≥ 20× yesterday. If they happen on rainy days, they are real
  storms on small, fast creeks; if they happen on dry days, they are suspicious.

In [8]:
flow_t = level_train[level_train["measure"] == "streamflow"]
gh_t   = level_train[level_train["measure"] == "gauge_height"]
checks = pd.DataFrame({
    "rows": [(flow_t["value"] < 0).sum(), (flow_t["value"] == 0).sum(), (gh_t["value"] < 0).sum()],
    "gauges": [flow_t.loc[flow_t["value"] < 0, "monitoring_location_id"].nunique(),
               flow_t.loc[flow_t["value"] == 0, "monitoring_location_id"].nunique(),
               gh_t.loc[gh_t["value"] < 0, "monitoring_location_id"].nunique()]},
    index=["negative streamflow (impossible)", "zero streamflow (creek dried up?)", "negative gauge height (normal)"])
display(checks)

,rows,gauges
negative streamflow (impossible),0,0
zero streamflow (creek dried up?),45,6
negative gauge height (normal),0,0


In [9]:
# (b) Stuck readings: runs of the exact same value on back-to-back days.
d = level_train.copy()
same_gauge = d["monitoring_location_id"].eq(d["monitoring_location_id"].shift())
next_day   = d["date"].diff().dt.days.eq(1)
same_value = d["value"].eq(d["value"].shift())
run_id = (~(same_gauge & next_day & same_value)).cumsum()
runs = d.groupby(run_id).agg(gauge=("monitoring_location_id", "first"), measure=("measure", "first"),
                             start=("date", "first"), days=("value", "size"), value=("value", "first"))
stuck = runs[runs["days"] >= STUCK_DAYS].sort_values("days", ascending=False)

print(f"Runs of {STUCK_DAYS}+ identical days: {len(stuck)} runs at {stuck['gauge'].nunique()} gauges, "
      f"covering {stuck['days'].sum():,} of {len(d):,} training rows ({stuck['days'].sum() / len(d):.2%})")
display(stuck.head(15))

Runs of 7+ identical days: 25 runs at 9 gauges, covering 238 of 171,099 training rows (0.14%)


,gauge,measure,start,days,value
54335,USGS-0214265828,streamflow,2023-10-20,24,0.03
54328,USGS-0214265828,streamflow,2023-09-21,19,0.02
54053,USGS-0214265828,streamflow,2022-10-14,17,0.02
103951,USGS-0214640410,gauge_height,2024-06-19,11,0.45
96882,USGS-02146315,streamflow,2023-10-20,11,0.23
58140,USGS-0214266080,streamflow,2024-10-15,10,0.75
104066,USGS-0214640410,gauge_height,2024-11-29,10,0.48
53762,USGS-0214265828,streamflow,2021-10-16,9,0.00
53968,USGS-0214265828,streamflow,2022-06-24,8,0.01
150374,USGS-021473415,streamflow,2022-09-22,8,0.00


In [10]:
# (c) Huge one-day jumps vs. rain. Each gauge gets the rain of its own weather grid square.
f = (flow_t.merge(cells[["monitoring_location_id", "cell_id"]], on="monitoring_location_id")
           .merge(train_weather[["cell_id", "date", "precipitation_sum"]], on=["cell_id", "date"], how="left")
           .sort_values(["monitoring_location_id", "date"]))
g = f.groupby("monitoring_location_id")
f["yesterday"] = g["value"].shift(1)
f["consecutive"] = g["date"].diff().dt.days.eq(1)
f["rain_2d"] = f["precipitation_sum"] + g["precipitation_sum"].shift(1).fillna(0)   # today + yesterday

jumps = f[f["consecutive"] & (f["yesterday"] > 0) & (f["value"] >= BIG_JUMP * f["yesterday"])]
rainy_share_jumps = (jumps["rain_2d"] >= RAINY_DAY_IN).mean()
rainy_share_all   = (f["rain_2d"] >= RAINY_DAY_IN).mean()

print(f"{BIG_JUMP}x+ one-day jumps: {len(jumps):,} at {jumps['monitoring_location_id'].nunique()} gauges")
print(f"Share of jumps on rainy days (>= {RAINY_DAY_IN} in. today+yesterday): {rainy_share_jumps:.0%}")
print(f"Share of ALL days that are rainy, for comparison:                     {rainy_share_all:.0%}")
print("\nJumps on DRY days (the only suspicious ones):")
display(jumps[jumps["rain_2d"] < RAINY_DAY_IN][["monitoring_location_id", "date", "yesterday", "value", "rain_2d"]].head(10))

20x+ one-day jumps: 1,623 at 64 gauges
Share of jumps on rainy days (>= 0.25 in. today+yesterday): 91%
Share of ALL days that are rainy, for comparison:                     28%

Jumps on DRY days (the only suspicious ones):


,monitoring_location_id,date,yesterday,value,rain_2d
8175,USGS-0212393300,2022-07-13,4.14,108.00,0.22
9274,USGS-02124080,2020-07-15,4.33,121.00,0.07
16213,USGS-02124269,2024-07-12,0.11,14.30,0.23
17310,USGS-0212427947,2022-07-13,0.32,15.10,0.15
17670,USGS-0212427947,2023-07-08,0.55,17.70,0.20
18040,USGS-0212427947,2024-07-12,0.13,5.30,0.11
19867,USGS-0212430653,2024-07-12,0.96,26.20,0.13
22797,USGS-0212467451,2022-07-19,3.18,355.00,0.01
24624,USGS-0212467595,2022-07-19,4.20,285.00,0.01
26853,USGS-02126000,2023-08-25,31.70,949.00,0.07


**How to read it:**
- (a) Any negative streamflow would be a real error. Zero flow at a few small creeks is plausible in dry summers.
- (b) Look at the `value` column of the longest runs. Tiny values (0.02 cfs) or low gauge heights in
  late summer/fall are most likely a real, very low, steady creek, not a broken sensor.
- (c) The key comparison is the two percentages. Only about 1 in 4 days is rainy overall, so if
  ~90% of the jumps land on rainy days, the jumps are real storms, not errors. Many of the
  "dry-day" jumps are in July — summer thunderstorms too small for the ~11 km weather grid to see.

**Decision 3:** remove nothing automatically. Keep jumps (real storms) and stuck runs (real low flow).
Only remove negative streamflow if any appears. Leave missing days missing — don't fill them in at this stage.

## 4. How different are the rivers? (training years only)

**Question:** can we compare raw readings across gauges, or do we need a per-gauge scale?

**Worked example:** if a tiny creek normally carries 3 cfs and the Catawba normally carries 3,000 cfs,
then "300 cfs" is a huge flood on the creek and a drought on the Catawba. Raw numbers can't share one model.

In [11]:
typical = (level_train.groupby(["monitoring_location_id", "measure"])["value"].median()
           .rename("median_value").reset_index()
           .merge(sites[["monitoring_location_id", "drainage_area"]], on="monitoring_location_id", how="left"))
flow_med = typical[typical["measure"] == "streamflow"]["median_value"]
print(f"Typical (median) streamflow across gauges: smallest {flow_med.min():,.2f} cfs, "
      f"largest {flow_med.max():,.0f} cfs  ->  {flow_med.max() / flow_med.min():,.0f}x apart")

typical["size_group"] = pd.cut(typical["drainage_area"], [0, 10, 100, 1000, 1e6],
                               labels=["tiny (<10 sq mi)", "small (10-100)", "medium (100-1,000)", "large (1,000+)"])
display(typical.groupby(["size_group", "measure"], observed=True)["median_value"]
        .agg(["count", "min", "median", "max"]))

Typical (median) streamflow across gauges: smallest 0.25 cfs, largest 6,580 cfs  ->  26,320x apart


count    min   median      max
size_group         measure                                     
tiny (<10 sq mi)   gauge_height      2   0.46     0.83     1.20
                   streamflow       24   0.25     2.36    11.90
small (10-100)     gauge_height      9   0.69     1.37     4.06
                   streamflow       35   4.47    13.70   108.00
medium (100-1,000) gauge_height      1   2.03     2.03     2.03
                   streamflow       10  47.30   187.00 1,230.00
large (1,000+)     gauge_height      3   4.25     6.58     8.20
                   streamflow        8 508.00 3,080.00 6,580.00

**How to read it:** the "x apart" number is the point — typical flows differ by thousands of times.
`drainage_area` (the land area that drains into the gauge) explains most of it.

**A per-gauge percentile fixes this.** Worked example: if a gauge's training readings sorted from
low to high put today's value above 85 of every 100 days, today's score is **85** — whatever the
size of the river. This also works for gauge-height gauges, because higher water at the *same*
gauge always means more water.

**Decision 4:** turn each gauge's level into a 0–100 percentile using **only that gauge's 2020–2024
readings** (test-period values get scored against the training history, not their own).

## 5. When do the extremes happen — and what should "drought" and "high water" be compared to? (training years only)

This section has four steps:

| Step | What it does |
|---|---|
| **5a. The problem** | Shows that our original labels are bunched into certain months. |
| **5b. The fix** | Explains a new rule: compare high water to *all days*, drought to *the same time of year*. |
| **5c. Build it** | Computes the new labels. |
| **5d. Check it** | Three checks that the new rule behaves the way it was designed to. |
| **5e. Check it against official maps** | Compares old and new drought labels with the U.S. Drought Monitor. |

### How every day is scored in this section: against the OTHER years

A score says "how high or low is today compared with this gauge's history." The history must **not**
include the day's own year.

**Why:** suppose 2022 was a very dry year. If 2022's days were ranked against a history that includes
2022 itself, its own dry days would fill the "bottom 10%," so at most ~10% of its days could ever be
labeled drought, and a long drought would be cut into short pieces. Ranked against the *other* years
(2020, 2021, 2023, 2024), a dry 2022 can show as dry as it really was — the same way a future day is
judged against the past.

| Day being scored | Compared against |
|---|---|
| a day in 2022 | 2020, 2021, 2023, 2024 |
| a day in 2020 | 2021, 2022, 2023, 2024 |

This is the same rule `build_training_data.py` uses, so the notebook and the pipeline agree.
A side effect: the share of days labeled drought or high water is no longer forced to exactly 10% —
dry and wet years now show up as such.

In [12]:
WINDOW_DAYS = 15   # +/- days around each date for the "same time of year" comparison (used in 5b-5c)

def day_of_year_365(dates):
    # Day of year 1-365 on a non-leap calendar (Feb 29 is treated as Feb 28), so years line up.
    md_text = pd.to_datetime(dates).dt.strftime("%m-%d").replace("02-29", "02-28")
    return pd.to_datetime("2021-" + md_text).dt.dayofyear.to_numpy()

def score_against(values, history):
    # 0-100: share of `history` below each value (ties count half).
    history = np.sort(history)
    if len(history) == 0:
        return np.full(len(values), np.nan)
    below = np.searchsorted(history, values, "left")
    at_or_below = np.searchsorted(history, values, "right")
    return (below + at_or_below) / 2 / len(history) * 100

def score_vs_other_years(df, value_col, window_days=None):
    # Score each row against the SAME gauge's values from the OTHER years.
    # window_days=None -> compare to all days of the other years (whole-year score).
    # window_days=15   -> compare only to days within +/-15 calendar days of this date (same time of year).
    doy = day_of_year_365(df["date"])
    year = pd.to_datetime(df["date"]).dt.year.to_numpy()
    vals = df[value_col].to_numpy(dtype=float)
    gauges = df["monitoring_location_id"].to_numpy()
    out = np.full(len(df), np.nan)
    for gid in np.unique(gauges):
        idx = np.where(gauges == gid)[0]
        v, d, yr = vals[idx], doy[idx], year[idx]
        has_value = ~np.isnan(v)
        for y in np.unique(yr):
            rows = has_value & (yr == y)                  # the days being scored
            other = has_value & (yr != y)                 # what they are compared against
            if window_days is None:
                out[idx[rows]] = score_against(v[rows], v[other])
                continue
            for day in range(1, 366):
                today = rows & (d == day)
                if not today.any():
                    continue
                dist = np.abs(d - day)
                dist = np.minimum(dist, 365 - dist)       # Dec 25 and Jan 5 are 11 days apart
                out[idx[today]] = score_against(v[today], v[other & (dist <= window_days)])
    return out

### 5a. The problem: the original labels follow the calendar

**Original rule (we'll call it A):** rank each day against *all* days of the gauge's other years.
- bottom 10% → **drought**
- top 10% → **high water**

If those labels were spread evenly through the year, every month would have about 10% of each.
The table below shows what actually happens.

In [13]:
lt = level_train.copy()
lt["month"] = lt["date"].dt.month
lt["pct"] = score_vs_other_years(lt, "value")          # rule A: vs ALL days of the other years

by_month_a = (lt.assign(drought=lt["pct"] < LOW_PCT, high_water=lt["pct"] >= HIGH_PCT)
                .groupby("month")[["drought", "high_water"]].mean().mul(100).round(1))
by_month_a.columns = ["% drought", "% high water"]
display(by_month_a)

,% drought,% high water
month,,
1,1.00,21.00
2,0.90,19.80
3,0.40,14.60
4,0.50,8.80
5,2.10,13.10
6,11.50,6.10
7,10.80,6.50
8,16.40,7.80
9,26.30,7.70


**What the table shows:**
- **Drought** is almost never labeled January–May (about 2% or less) but ~34% of October days are "drought."
- **High water** is the mirror image: ~20% of January days, ~3% of October days.

**Why:** Carolina rivers are *always* low in late summer and fall (heat and trees pull water out of the
ground) and *always* higher in winter. So under rule A, "drought" mostly means "it's autumn."

**Why that's a problem for drought, but not for high water:**
- **High water is about how much water there is.** A river in its top 10% floods the same roads in
  January as in October. That winter floods are more common is a real fact the app *should* reflect.
  (NWS flood stages are fixed heights for the same reason.)
- **Drought is about missing water — less than expected for this time of year.** A normal October
  is low every year; that's the dry season, not a drought. A model trained on rule A could just
  learn "October → drought" without looking at the river.

### 5b. The fix: compare each side the right way (rule B)

This split was recommended by a resource education specialist at the county soil and water district.

| Label | Compared to | Uses | Why |
|---|---|---|---|
| **High water** | all of the gauge's days (same as A) | the **daily** level | flooding depends on the amount of water |
| **Drought** | the gauge's days at the **same time of year** | the **7-day average** level (Decision 7) | drought = less than normal *for the season* |

**What "same time of year" means: a ±15-day window.**
For each date, the comparison days are every day within 15 calendar days of that date, in the other years.

**Worked example — October 20:**
- Window = October 5 to November 4 (31 days) in each of the *other* years — for Oct 20, 2022: 2020, 2021, 2023, 2024
- That's about 31 × 4 = **124 comparison days** (in the training data for 2025+, all 5 years: ~155)
- If this October 20's 7-day average is lower than 95 of every 100 of those days → it's in the bottom 10% → **drought**

**Why a window instead of the calendar month:** with calendar months, Oct 31 is compared only to
October days and Nov 1 only to November days, so the drought cut-off jumps overnight. The window slides
one day at a time, so the cut-off changes smoothly. Step 5d shows this for one gauge.

**Tie rule:** a day could in theory be both (top 10% of all days *and* bottom 10% for its season).
Those days are labeled **high water**, because a flood is the more urgent warning.

### 5c. Build the new labels

In [14]:
lt = lt.sort_values(["monitoring_location_id", "date"]).reset_index(drop=True)
lt["value_7d"] = lt.groupby("monitoring_location_id")["value"].transform(lambda s: s.rolling(7, min_periods=5).mean())
lt["pct_7d_season"] = score_vs_other_years(lt, "value_7d", WINDOW_DAYS)   # rule B drought; takes ~30 seconds

lt["high_b"]    = lt["pct"] >= HIGH_PCT                                  # vs all days, daily level
lt["both_b"]    = lt["high_b"] & (lt["pct_7d_season"] < LOW_PCT)         # would be both -> tie rule
lt["drought_b"] = (lt["pct_7d_season"] < LOW_PCT) & ~lt["high_b"]        # vs same season, 7-day level
lt["drought_a"] = lt["pct"] < LOW_PCT                                    # original, for comparison

print(f"Days that would be both (labeled high water by the tie rule): {lt['both_b'].sum():,} of {len(lt):,}")

Days that would be both (labeled high water by the tie rule): 22 of 171,099


### 5d. Check the new rule

**Check 1 — does drought stop following the calendar?** Under rule B, drought should be spread fairly
evenly across the months instead of piling up in the fall. It won't be exactly 10% per month: days are
compared against the *other* years, so a month that was drier than usual across 2020–2024 shows more
drought. Compare the spread, not the exact values. High water should be unchanged from A.

In [15]:
by_month_b = lt.groupby("month")[["drought_b", "high_b"]].mean().mul(100).round(1)
by_month_b.columns = ["% drought", "% high water"]
display(pd.concat({"A: original": by_month_a, "B: new rule": by_month_b}, axis=1))

A: original              B: new rule             
        % drought % high water   % drought % high water
month                                                  
1            1.00        21.00       11.70        21.00
2            0.90        19.80       15.10        19.80
3            0.40        14.60       11.90        14.60
4            0.50         8.80       10.00         8.80
5            2.10        13.10       14.60        13.10
6           11.50         6.10       16.00         6.10
7           10.80         6.50       11.70         6.50
8           16.40         7.80       14.70         7.80
9           26.30         7.70       12.70         7.70
10          34.30         4.20       16.10         4.20
11          21.90         7.30       14.00         7.30
12           9.70        11.70       16.00        11.70

**Check 2 — does it still find the genuinely dry years?** This is the important one. Rule B forces each
*season* to average ~10% across the 5 years, but it does **not** force each *year* to 10%. If a drought
year still stands out, B is detecting real droughts, not just relabeling the calendar.

In [16]:
lt["year"] = lt["date"].dt.year
by_year = pd.DataFrame({
    "A: % drought": lt.groupby("year")["drought_a"].mean() * 100,
    "B: % drought": lt.groupby("year")["drought_b"].mean() * 100,
    "% high water (same in A and B)": lt.groupby("year")["high_b"].mean() * 100,
}).round(1)
display(by_year)

both = (lt["drought_a"] & lt["drought_b"]).sum()
print(f"Drought days - A: {lt['drought_a'].sum():,}  |  B: {lt['drought_b'].sum():,}  |  in both: {both:,} "
      f"({both / lt['drought_a'].sum():.0%} of A's drought days are still drought under B)")

,A: % drought,B: % drought,% high water (same in A and B)
year,,,
2020,0.70,1.70,19.60
2021,14.40,11.70,9.10
2022,19.80,28.20,5.60
2023,17.50,17.70,8.10
2024,4.20,9.30,11.00


Drought days - A: 19,418  |  B: 23,461  |  in both: 10,246 (53% of A's drought days are still drought under B)


**Check 3 — does the window remove the month-boundary jump?** For one example gauge, the table shows the
**drought cut-off** (the 7-day average flow below which a day counts as drought) for each date around
Oct 31 / Nov 1, computed two ways: by calendar month vs. by the ±15-day window.

In [17]:
# Example gauge: the streamflow gauge with the median drainage area.
flow_gauges = sites[sites["monitoring_location_id"].isin(keep[keep == "streamflow"].index)].dropna(subset=["drainage_area"])
example = flow_gauges.sort_values("drainage_area").iloc[len(flow_gauges) // 2]
ex = lt[lt["monitoring_location_id"] == example["monitoring_location_id"]].dropna(subset=["value_7d"]).copy()
ex["doy"] = day_of_year_365(ex["date"])

boundary_dates = pd.date_range("2021-10-25", "2021-11-07")
rows = []
for day in boundary_dates:
    d = day.dayofyear
    dist = np.minimum(np.abs(ex["doy"] - d), 365 - np.abs(ex["doy"] - d))
    rows.append({"date": day.strftime("%b %d"),
                 "cut-off: calendar month (cfs)": ex.loc[ex["month"] == day.month, "value_7d"].quantile(LOW_PCT / 100),
                 "cut-off: +/-15-day window (cfs)": ex.loc[dist <= WINDOW_DAYS, "value_7d"].quantile(LOW_PCT / 100)})
print(f"Example gauge: {example['monitoring_location_name']} ({example['drainage_area']:,.0f} sq mi)")
display(pd.DataFrame(rows).set_index("date").round(2))

Example gauge: CODDLE CR AT SR 1612 NEAR DAVIDSON, NC (23 sq mi)


,cut-off: calendar month (cfs),cut-off: +/-15-day window (cfs)
date,,
Oct 25,4.30,4.44
Oct 26,4.30,4.44
Oct 27,4.30,4.52
Oct 28,4.30,4.55
Oct 29,4.30,4.55
Oct 30,4.30,4.55
Oct 31,4.30,4.55
Nov 01,5.29,4.55
Nov 02,5.29,4.56


**How to read Check 3:** in the calendar-month column the cut-off stays flat all of October, then jumps
on Nov 1. A river at the same level could be "drought" on Oct 31 and "normal" on Nov 1 (or the reverse)
for no real reason. The window column changes a little each day instead.

### 5e. Check against the official drought maps (U.S. Drought Monitor)

Checks 1–3 only show the new rule behaves the way it was designed to. This check compares it to an
**outside** source: the U.S. Drought Monitor (USDM), the official weekly drought map.

**What USDM gives us** (`data/validation/usdm_county_weekly.csv`, made earlier by `validate_labels.py`):
for every county and every week, the worst drought category covering at least half the county:

| Category | Meaning |
|---|---|
| none | normal |
| D0 | abnormally dry (not yet drought) |
| D1 | moderate drought |
| D2 | severe drought |
| D3–D4 | extreme / exceptional drought |

Like rule B, USDM judges dryness against what's normal **for that time of year**, so it is a fair referee.

**Method in plain words:**
1. For each county and each USDM week (Tuesday–Monday), take all of our gauges in that county.
2. Compute the **% of those gauge-days labeled drought** — once under rule A (original), once under rule B (new).
3. Compare those percentages with what USDM said that week.

**Worked example:** Mecklenburg has, say, 40 kept gauges. In one week that's 40 × 7 = 280 gauge-days. If 70
of them are labeled drought, that county-week scores 70 ÷ 280 = **25%**.

**Limits to keep in mind:** USDM is not perfect "ground truth." It blends rain, soil moisture, streamflow and
reported impacts over weeks to months, and compares against decades of history (we have 5 years). A single
fast creek can recover after one storm while its county stays in drought.

In [18]:
from src.flowcast.model.validate_labels import COUNTY_FIPS   # (state, county) -> 5-digit county code

try:
    usdm = read_dataset("usdm_county_weekly", dtype={"fips": str})
except Exception as e:                       # e.g. never uploaded to S3 -> use the local copy
    print(f"Could not read from {SOURCE} ({type(e).__name__}); using data/validation/usdm_county_weekly.csv")
    usdm = read_dataset("usdm_county_weekly", source="local", dtype={"fips": str})
usdm["valid_start"] = pd.to_datetime(usdm["valid_start"])
usdm = usdm[usdm["valid_start"] < SPLIT_DATE]                  # training years only

gauge_county = sites[["monitoring_location_id", "state_name", "county_name"]].copy()
gauge_county["fips"] = [COUNTY_FIPS.get((s, c)) for s, c in zip(gauge_county["state_name"], gauge_county["county_name"])]

v = lt.merge(gauge_county[["monitoring_location_id", "fips"]], on="monitoring_location_id")
v["week_start"] = v["date"] - pd.to_timedelta((v["date"].dt.weekday - 1) % 7, unit="D")   # back to Tuesday
weekly = (v.groupby(["fips", "week_start"])
           .agg(pct_drought_A=("drought_a", "mean"), pct_drought_B=("drought_b", "mean"))
           .mul(100).reset_index()
           .merge(usdm[["fips", "valid_start", "usdm_category", "dsci"]],
                  left_on=["fips", "week_start"], right_on=["fips", "valid_start"]))
weekly["usdm_group"] = weekly["usdm_category"].replace({"D4": "D3"})   # D4 is very rare: group with D3
print(f"County-weeks compared: {len(weekly):,}")

  reading s3://flowcast-pred-app-data/processed/validation/2026-09-27/usdm_county_weekly.csv
County-weeks compared: 4,192


**Check 4a — does the % labeled drought rise as official drought gets worse?** A good label should be near
0% when USDM says "none" and climb steadily from D0 to D3.

In [19]:
order = ["none", "D0", "D1", "D2", "D3"]
display(weekly.groupby("usdm_group")
        .agg(county_weeks=("fips", "size"),
             **{"A: % gauge-days drought": ("pct_drought_A", "mean"),
                "B: % gauge-days drought": ("pct_drought_B", "mean")})
        .reindex(order).round(1))
print("Correlation with USDM's drought severity index (DSCI, 0 = none ... 500 = all D4; higher = better match):")
print(f"  A: {weekly['pct_drought_A'].corr(weekly['dsci']):.2f}   B: {weekly['pct_drought_B'].corr(weekly['dsci']):.2f}")

,county_weeks,A: % gauge-days drought,B: % gauge-days drought
usdm_group,,,
none,2917,7.00,8.50
D0,836,17.10,22.20
D1,296,23.80,30.90
D2,137,39.80,53.30
D3,6,71.90,81.00


Correlation with USDM's drought severity index (DSCI, 0 = none ... 500 = all D4; higher = better match):
  A: 0.34   B: 0.41


**Check 4b — hits and false alarms.** Call a county-week "our drought week" if **at least half** its
gauge-days are labeled drought. Then ask three questions:
- **Caught:** of the weeks USDM said D1 or worse, what share did we also flag?
- **Right when flagged:** of the weeks we flagged, what share did USDM call at least abnormally dry (D0+)?
- **False alarms by month:** weeks we flagged while USDM said "none." Rule A's problem predicts these will
  pile up in late summer and fall.

In [20]:
weekly["usdm_d1_plus"] = weekly["usdm_category"].isin(["D1", "D2", "D3", "D4"])
weekly["usdm_d0_plus"] = weekly["usdm_category"] != "none"
weekly["month"] = weekly["week_start"].dt.month

rows, false_alarms = {}, {}
for rule in ["A", "B"]:
    flagged = weekly[f"pct_drought_{rule}"] >= 50
    rows[f"{rule}"] = {
        "weeks flagged": flagged.sum(),
        "caught: % of USDM D1+ weeks we flagged": (flagged & weekly["usdm_d1_plus"]).sum() / weekly["usdm_d1_plus"].sum() * 100,
        "right when flagged: % USDM D0+": (flagged & weekly["usdm_d0_plus"]).sum() / flagged.sum() * 100,
        "false alarms (USDM said none)": (flagged & ~weekly["usdm_d0_plus"]).sum(),
    }
    false_alarms[rule] = weekly[flagged & ~weekly["usdm_d0_plus"]].groupby("month").size()
display(pd.DataFrame(rows).T.round(1))
print("False alarms by month (USDM said 'none' but we flagged drought):")
display(pd.DataFrame(false_alarms).reindex(range(1, 13)).fillna(0).astype(int).T)

,weeks flagged,caught: % of USDM D1+ weeks we flagged,right when flagged: % USDM D0+,false alarms (USDM said none)
A,406.00,26.40,59.40,165.00
B,531.00,38.50,64.60,188.00


False alarms by month (USDM said 'none' but we flagged drought):


month,1,2,3,4,5,6,7,8,9,10,11,12
A,0,0,0,0,0,11,14,39,61,37,3,0
B,1,11,34,11,38,20,13,36,19,5,0,0


**How to read Check 4:**
- **4a:** both rules should climb from "none" to D3. The rule that climbs more steeply, with the higher
  correlation, matches USDM better overall.
- **4b:** compare the two rows. A higher "caught" and a higher "right when flagged" are better; more false
  alarms are worse. Then look at the month table: if rule A's false alarms sit almost entirely in
  June–October and rule B's are spread through the year, rule A was flagging the normal dry season.
- **Low "caught" for both rules is expected**, not a bug: USDM keeps a county in drought for weeks or months,
  while a creek's flow bounces back after a single storm. Our labels describe the river this week; USDM
  describes the county's longer-term water situation.

**What to say in the write-up:** report both rules' numbers side by side and claim only what the tables show.
At the time of writing (2020–2024): rule B matched USDM better overall (higher correlation, more drought weeks
caught, more of its flags right). It raised somewhat *more* false alarms than A, but they were spread through
the year instead of concentrated in the dry season.

**Decision 5:**
- **High water:** daily level, ranked against **all** of the gauge's 2020–2024 days (top 10%).
- **Drought:** 7-day average level, ranked against the gauge's 2020–2024 days **within ±15 calendar days** (bottom 10%).
- **Tie:** label high water.
- **Consequence:** there are now **two** scores per day (one for each side), not one 0–100 number. The 3-level
  classifier is built from both, and the single 0–100 regression model is dropped.

**Trade-off to state in the write-up:** each date's drought cut-off rests on only ~124-155 days from 4-5 years,
so one unusually dry year shifts it.

## 6. How fast does rain reach the river? (training years only)

**Question:** after a big storm, how many days until the river peaks? This tells us (1) which rain
windows matter as features and (2) whether predicting 3 days ahead is realistic without a rain forecast.

**Method in plain words:** find every "big storm" day at a gauge (≥ 1 inch of rain in its grid square,
with < 0.25 in. in the 3 days before, so storms don't overlap). Look at the river that day and the next
7 days. The day with the highest reading is the peak. "Lag 0" = peaked the same day, "lag 1" = next day, etc.

In [21]:
# Wide tables (rows = dates, columns = gauges) make "look 0-7 days ahead" a simple shift.
dates = pd.date_range(TRAIN_START, SPLIT_DATE - pd.Timedelta(days=1))
lvl_w = level_train.pivot(index="date", columns="monitoring_location_id", values="value").reindex(dates)
rain_by_gauge = (cells[["monitoring_location_id", "cell_id"]]
                 .merge(train_weather[["cell_id", "date", "precipitation_sum"]], on="cell_id")
                 .pivot(index="date", columns="monitoring_location_id", values="precipitation_sum"))
rain_w = rain_by_gauge.reindex(index=dates, columns=lvl_w.columns)

prior3 = rain_w.shift(1).rolling(3, min_periods=1).sum()
storm = (rain_w >= BIG_RAIN_IN) & (prior3 < RAINY_DAY_IN)

window = np.stack([lvl_w.shift(-k).to_numpy() for k in range(8)])   # shape: 8 x dates x gauges
enough = (~np.isnan(window)).sum(axis=0) >= 6                       # need >= 6 of the 8 days
peak_lag = np.argmax(np.nan_to_num(window, nan=-np.inf), axis=0)

mask = storm.to_numpy() & enough
lags = pd.DataFrame({"monitoring_location_id": np.tile(lvl_w.columns, (len(dates), 1))[mask],
                     "lag_days": peak_lag[mask]})
lags = lags.merge(sites[["monitoring_location_id", "drainage_area"]], on="monitoring_location_id", how="left")
lags["size_group"] = pd.cut(lags["drainage_area"], [0, 10, 100, 1000, 1e6],
                            labels=["tiny (<10 sq mi)", "small (10-100)", "medium (100-1,000)", "large (1,000+)"])

print(f"Storms analyzed: {len(lags):,} (gauge-storm pairs)")
display(pd.crosstab(lags["size_group"], lags["lag_days"], normalize="index").mul(100).round(0)
        .add_suffix(" d").assign(storms=lags["size_group"].value_counts()))

Storms analyzed: 2,214 (gauge-storm pairs)


lag_days,0 d,1 d,2 d,3 d,4 d,5 d,6 d,7 d,storms
size_group,,,,,,,,,
tiny (<10 sq mi),60.00,18.00,3.00,7.00,3.00,2.00,1.00,6.00,599
small (10-100),43.00,33.00,4.00,7.00,4.00,4.00,1.00,4.00,1028
"medium (100-1,000)",11.00,50.00,14.00,8.00,6.00,6.00,2.00,3.00,283
"large (1,000+)",18.00,19.00,21.00,10.00,10.00,8.00,6.00,8.00,250


**How to read it:** each row adds up to 100%. Look at where most of each row's percentage sits.
Small creeks peak the same day or next day; only the large rivers spread out over 2–4 days.

**What this means for a 3-day-ahead prediction:** for most of our gauges, the river reacts to rain
within ~1 day. So today's river reading says little about a storm 3 days from now — the prediction
depends mostly on the **rain forecast**. That is why `precip_next_3d` dominated the old models, and
why using *actual* rain in training overstated accuracy.

**Decision 6:**
- Rain features: keep short windows (rain today, last 3 days) for the fast creeks and a 7-day window
  for big rivers; longer windows (30 days) only for drought/dryness.
- **Horizons are set separately for each side** (decided after this EDA; see section 9):
  - **High water starts at 1 day ahead**, where today's river reading still carries real information.
    3 days ahead stays available as a setting.
  - **Drought starts at 7 days ahead.** Shorter drought horizons are not used: the drought label is a 7-day
    average, so 1–3 days ahead would mostly "predict" days that are already known. 14 days stays available
    as a setting.
- **Future rain:** high-water models are trained and reported **both with and without** the rain that
  actually fell over the horizon (best case = perfect forecast, worst case = no forecast). Drought models
  get **no** future rain, because the app's forecast doesn't reach that far reliably.

## 7. How long do extremes last? (training years only)

**Question:** when a gauge enters drought or high water, how many days in a row does it stay there?

**Why it matters:** if most "drought" stretches last 1 day, the label is flickering on and off, not
describing a real drought (which builds over weeks). A 7-day average flow smooths that out.

**Worked example of a run:** labels for 6 days = normal, drought, drought, drought, normal, drought →
two drought runs, lasting 3 days and 1 day.

In [22]:
def run_lengths(df, flag_col):
    # Lengths of back-to-back True stretches in flag_col, per gauge.
    same_gauge = df["monitoring_location_id"].eq(df["monitoring_location_id"].shift())
    next_day = df["date"].diff().dt.days.eq(1)
    same_flag = df[flag_col].eq(df[flag_col].shift())
    rid = (~(same_gauge & next_day & same_flag)).cumsum()
    return df[df[flag_col]].groupby(rid[df[flag_col]]).size()

lt = lt.sort_values(["monitoring_location_id", "date"])
lt["value_7d"] = lt.groupby("monitoring_location_id")["value"].transform(lambda s: s.rolling(7, min_periods=5).mean())
lt["pct_7d"] = score_vs_other_years(lt, "value_7d")      # 7-day average vs all days of the other years

lt["drought_daily"] = lt["pct"] < LOW_PCT
lt["drought_7d"]    = lt["pct_7d"] < LOW_PCT
lt["high_daily"]    = lt["pct"] >= HIGH_PCT

rows = {}
for name, col in [("drought (daily value, all days)", "drought_daily"),
                  ("drought (7-day average, all days)", "drought_7d"),
                  ("drought (7-day average, +/-15-day season = Decision 5)", "drought_b"),
                  ("high water (daily value, all days)", "high_daily")]:
    L = run_lengths(lt, col)
    rows[name] = {"runs": len(L), "median days": L.median(), "90th pct days": L.quantile(0.9),
                  "% of runs only 1 day": (L == 1).mean() * 100,
                  "% of labeled days in runs of 7+": L[L >= 7].sum() / L.sum() * 100}
display(pd.DataFrame(rows).T.round(1))

,runs,median days,90th pct days,% of runs only 1 day,% of labeled days in runs of 7+
"drought (daily value, all days)","3,506.00",3.00,14.00,27.50,66.40
"drought (7-day average, all days)","1,706.00",7.00,26.00,9.80,87.20
"drought (7-day average, +/-15-day season = Decision 5)","2,803.00",5.00,19.00,13.50,79.20
"high water (daily value, all days)","9,089.00",1.00,4.00,54.50,17.30


**How to read it:** compare the drought rows. Fewer runs, a longer median, and fewer 1-day runs mean the
label flickers less.
- Row 1 → row 2: the 7-day average roughly halves the number of runs and cuts 1-day runs from ~28% to ~10%.
- Row 3 is the drought label chosen in Decision 5. Its runs are a bit **shorter** than row 2's: comparing to
  the season removes the long "every autumn is low" stretches, leaving periods of being *unusually* low for
  the time of year. It still flickers much less than row 1 (the original daily label).
- The high-water row shows short runs (median ~1 day): high water is a brief spike, so averaging it over
  7 days would erase it.

**Open question — 7-day vs 30-day average for drought.** With this scoring, a 30-day average (same rule B
comparison) matched the U.S. Drought Monitor better than the 7-day average: it caught ~49% of official
drought weeks vs ~39%, was right when flagged ~69% vs ~65%, correlation 0.46 vs 0.41, with somewhat more false
alarms (214 vs 188), and its drought stretches last longer (median 10 days vs 5). The catch: a drought label
built from a 30-day average, predicted 7 days ahead, would already be ~23 of its 30 days known on the day
of the prediction — so the prediction would be mostly known in advance. A 30-day label only makes sense
with a longer horizon (e.g. 30 days ahead). The pipeline keeps the 7-day average until the team decides.

**Decision 7:** drought labels from the **7-day average** level; high-water labels from the **daily** level.

## 8. Weather data check (training years only)

**Question:** is the weather complete, in sensible ranges, and are any columns repeats of each other?

In [23]:
wcols = ["precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours",
         "et0_fao_evapotranspiration", "temperature_2m_max", "temperature_2m_min"]
print(f"Missing weather values: {train_weather[wcols].isna().sum().sum()}  |  "
      f"duplicate cell-days: {train_weather.duplicated(['cell_id', 'date']).sum()}")
display(train_weather[wcols].describe().T[["min", "mean", "max"]])
print("Correlation between pairs that might be repeats (1.00 = identical information):")
display(pd.Series({
    "precipitation_sum vs rain_sum": train_weather["precipitation_sum"].corr(train_weather["rain_sum"]),
    "temperature max vs min": train_weather["temperature_2m_max"].corr(train_weather["temperature_2m_min"]),
    "% of days with any snow": (train_weather["snowfall_sum"] > 0).mean() * 100}).round(2).to_frame("value"))

Missing weather values: 0  |  duplicate cell-days: 0


,min,mean,max
precipitation_sum,0.00,0.13,5.49
rain_sum,0.00,0.13,5.49
snowfall_sum,0.00,0.00,6.31
precipitation_hours,0.00,3.42,24.00
et0_fao_evapotranspiration,0.01,0.13,0.32
temperature_2m_max,23.80,71.78,103.00
temperature_2m_min,1.20,52.99,79.10


Correlation between pairs that might be repeats (1.00 = identical information):


,value
precipitation_sum vs rain_sum,1.00
temperature max vs min,0.93
% of days with any snow,0.92


**How to read it:** precipitation and rain nearly identical, and snow on only a few days a year,
means those columns add little. Max and min temperature move together.

**Decision 8:** use `precipitation_sum` (drop `rain_sum`), drop `snowfall_sum`, keep one temperature
(`temperature_2m_max`) plus evapotranspiration (`et0`, how much water the air and plants pull out —
it drives drought).

## 9. Summary of decisions → the spec for the new training data and models

Two tables:
- **Table 1** — decisions that come straight from the data in sections 1–8.
- **Table 2** — the modeling plan the team settled after reviewing this notebook.

The gauge list with the chosen measure is saved to `data/eda/gauge_decisions.csv`, so the new
training-data script reads it instead of re-deciding.

**How the winner is picked (decided *before* seeing any results, so we can't pick the metric that
flatters our favorite model):**
- **Main metric: F1 score** for the "yes" class (high water, or drought). It balances two things:
  - **caught** (recall) — of the real high-water days, what share did the model flag?
  - **right when flagged** (precision) — of the days the model flagged, what share were really high water?
- **Worked example:** a model catches 60% of high-water days, and 40% of its alarms are right.
  F1 = 2 × 0.6 × 0.4 ÷ (0.6 + 0.4) = **0.48**. Plain accuracy is not used: on a rare event, always saying
  "no" scores ~90% accuracy and catches nothing.
- **Winner rule:** a model must beat the baseline on F1 in **every** cross-validation round, not just on
  average. Between the decision tree and the boosted model, keep the tree unless boosting wins by a clear
  margin (**≥ 0.05 F1**) — being able to explain the model is worth more than a small gain.

In [24]:
decisions = pd.DataFrame([
    ("1", "Which readings", "Keep provisional and qualified readings."),
    ("2", "Which gauges / measure", f"Streamflow if >= {MIN_COMPLETE_PCT}% complete in 2020-2024, else gauge height, else drop."),
    ("3", "Cleaning", "Remove nothing automatically (jumps = storms, stuck = low flow). Leave missing days missing."),
    ("4", "Common scale", "Per-gauge percentiles. Each day is compared against years it is not part of (training day: the other training years; test day: all training years)."),
    ("5", "Labels (rule B)", "High water: daily level vs ALL days (top 10%). Drought: 7-day avg vs same +/-15-day season (bottom 10%). Ties -> high water."),
    ("6", "Rain features", "Rain today, last 3 days, 7 days; 30 days for drought/dryness."),
    ("7", "Smoothing", "Drought from 7-day average level; high water from daily level. OPEN: 30-day average matched USDM better but needs a longer horizon (section 7)."),
    ("8", "Weather columns", "precipitation_sum, et0, temperature_2m_max (drop rain_sum, snowfall_sum, temperature_2m_min)."),
], columns=["#", "topic", "decision"]).set_index("#")
print("Table 1 - decisions from the data")
display(decisions)

plan = pd.DataFrame([
    ("A", "Model design", "Separate yes/no models per side and horizon (replaces the single 3-level model)."),
    ("A", "Starting models", "High water 1 day ahead; drought 7 days ahead. Settings allow high water 3 days and drought 14 days."),
    ("B", "Future rain", "High water: train and report WITH and WITHOUT actual future rain. Drought: never uses future rain."),
    ("C", "7-level classifier", "Dropped (also the single 0-100 regression model)."),
    ("D", "Cut-offs", "Keep bottom 10% = drought, top 10% = high water."),
    ("E", "Feature list", "To be drafted in the new training-data script for team review."),
    ("-", "Models compared", "Baseline ('same as today'), decision tree, boosted trees (scikit-learn HistGradientBoostingClassifier)."),
    ("-", "Main metric", "F1 for the 'yes' class; also report caught (recall) and right-when-flagged (precision)."),
    ("-", "Winner rule", "Must beat baseline F1 in every CV round; keep the tree unless boosting wins by >= 0.05 F1."),
    ("-", "Train / test", "Train 2020-2024, test 2025 onward; gap between them = that model's horizon. Same 4-round date-based CV."),
], columns=["#", "topic", "decision"]).set_index("#")
print("Table 2 - modeling plan (decided after the EDA)")
display(plan)

gauge_decisions = (comp.round(1).reset_index()
                   .merge(sites[["monitoring_location_id", "monitoring_location_name", "county_name", "drainage_area"]],
                          on="monitoring_location_id", how="left"))
out = Path("data/eda"); out.mkdir(parents=True, exist_ok=True)
gauge_decisions.to_csv(out / "gauge_decisions.csv", index=False)
print(f"Saved {len(gauge_decisions)} gauges to {out / 'gauge_decisions.csv'} "
      f"({(gauge_decisions['use_measure'] != 'drop').sum()} kept)")

Table 1 - decisions from the data


,topic,decision
#,,
1,Which readings,Keep provisional and qualified readings.
2,Which gauges / measure,"Streamflow if >= 90% complete in 2020-2024, else gauge height, else drop."
3,Cleaning,"Remove nothing automatically (jumps = storms, stuck = low flow). Leave missing days missing."
4,Common scale,Per-gauge percentiles. Each day is compared against years it is not part of (training day: the other training years; test day: all training years).
5,Labels (rule B),High water: daily level vs ALL days (top 10%). Drought: 7-day avg vs same +/-15-day season (bottom 10%). Ties -> high water.
6,Rain features,"Rain today, last 3 days, 7 days; 30 days for drought/dryness."
7,Smoothing,Drought from 7-day average level; high water from daily level. OPEN: 30-day average matched USDM better but needs a longer horizon (section 7).
8,Weather columns,"precipitation_sum, et0, temperature_2m_max (drop rain_sum, snowfall_sum, temperature_2m_min)."


Table 2 - modeling plan (decided after the EDA)


,topic,decision
#,,
A,Model design,Separate yes/no models per side and horizon (replaces the single 3-level model).
A,Starting models,High water 1 day ahead; drought 7 days ahead. Settings allow high water 3 days and drought 14 days.
B,Future rain,High water: train and report WITH and WITHOUT actual future rain. Drought: never uses future rain.
C,7-level classifier,Dropped (also the single 0-100 regression model).
D,Cut-offs,"Keep bottom 10% = drought, top 10% = high water."
E,Feature list,To be drafted in the new training-data script for team review.
-,Models compared,"Baseline ('same as today'), decision tree, boosted trees (scikit-learn HistGradientBoostingClassifier)."
-,Main metric,F1 for the 'yes' class; also report caught (recall) and right-when-flagged (precision).
-,Winner rule,Must beat baseline F1 in every CV round; keep the tree unless boosting wins by >= 0.05 F1.


Saved 102 gauges to data\eda\gauge_decisions.csv (94 kept)
